# 🔬 Notebook 04: 3D ViT + Multi-Scale FPN From-Scratch Ablation
### Scientific Representation Learning Baseline (Zero Pre-Training)

This dedicated notebook runs the **3D ViT + Multi-Scale FPN Trained From Scratch** ablation baseline:
1. **Scientific Question**: Isolates whether 3D VisReg JEPA's volumetric segmentation performance is driven by self-supervised representation learning ($\Delta_{\text{JEPA}}$) or purely by the high parameter capacity and multi-scale inductive bias of the 3D ViT-FPN architecture.
2. **Zero Pre-Trained Weights Required**: Model is initialized completely from random weights with a deterministic seed (`--from_scratch`). No pre-trained checkpoint upload is needed!
3. **Supervised 30-Epoch Training**: Trains 3D ViT-FPN end-to-end using combined Soft Dice and Binary Cross-Entropy (BCE) loss.
4. **Held-Out Test Set Evaluation**: Measures exact 3D Dice, IoU, 95th Percentile Hausdorff Distance (mm), and inference latency.
5. **Artifact Export**: Packages checkpoints and metrics into `vit_scratch_outputs.zip` for 1-click download and local master comparison.

> **Estimated Runtime**: ~1.2 - 1.8 hours on NVIDIA Tesla T4 GPU (accelerated by the $18.7\times$ fast validation engine).


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 4          # Parallel CPU data loading workers (Kaggle T4 provides 4 vCPUs)
BATCH_SIZE = 2           # Volumetric batch size for 3D ViT-FPN downstream training

# Belt-and-suspenders: pin the mounted full-pool dataset so every script
# resolving the default name finds it even without an explicit data_dir.
import os as _os
from pathlib import Path as _Path
_kaggle_full = _Path("/kaggle/input/brats-3d-full/brats_gli_3d_full")
if _kaggle_full.is_dir() and (_kaggle_full / "metadata.csv").exists():
    _os.environ["BRATS3D_DATA_DIR"] = str(_kaggle_full)
    print(f"Dataset env override: BRATS3D_DATA_DIR={_kaggle_full}")


## 2. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")


## 3. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist across Kaggle input mounts. If preprocessed data is absent, checks for raw BraTS data and automatically invokes `prepare_data_3d.py`.


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d_full")
meta_path = get_metadata_path("brats_gli_3d_full")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Checking for raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers {NUM_WORKERS} --seed {SEED}
    meta_path = get_metadata_path("brats_gli_3d_full")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(data_dir=data_dir, split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-full' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    )


## 5. Supervised 3D ViT-FPN Training From Scratch (30 Epochs, AMP)
Trains the 3D Vision Transformer backbone + Multi-Scale FPN decoder **from random initialization** without loading any pre-trained SSL weights (`--from_scratch`).

- **Encoder**: 3D ViT (Patch size $16^3$, Embedding dim $384$, Depth $8$, Heads $6$)
- **Decoder**: Multi-Scale FPN ($L_2, L_4, L_6, L_8$ progressive upsampling)
- **Loss**: Combined Soft Dice + BCE Loss
- **Validation Speed**: Fast 2.4s validation per epoch via decoupled HD95 calculation.


In [ ]:
!python scripts/train_downstream_3d.py \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --from_scratch \
    --seed {SEED} \
    --epochs 30 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate 3e-4 \
    --weight_decay 1e-4 \
    --amp


## 6. Held-Out Test Split Evaluation
Evaluates the trained from-scratch 3D ViT-FPN model on the held-out test split ($N=242$ volumes) across:
- Exact Volumetric 3D Dice Score (%)
- Volumetric 3D IoU (%)
- 95th Percentile Hausdorff Distance (HD95, mm)
- Inference Latency (ms per volume)


In [ ]:
!python scripts/evaluate_3d.py \
    --model_type visreg_jepa \
    --decoder_type multiscale \
    --from_scratch \
    --seed {SEED} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp \
    --tta


## 7. Low-Data Volumetric Label Efficiency for 3D ViT-FPN (From Scratch)
Evaluates fine-tuning performance across annotation budgets: $1\%$ ($11$ vols), $5\%$ ($57$ vols), $10\%$ ($114$ vols), $25\%$ ($286$ vols), $50\%$ ($572$ vols), and $100\%$ ($1,144$ vols) when trained from random initialization. This directly isolates the empirical representation gain of self-supervised JEPA pre-training.


In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --model_type visreg_jepa \
    --from_scratch \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --seed {SEED} \
    --epochs 15 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp


## 8. Out-of-Distribution (OOD) Scanner Shift Robustness for 3D ViT-FPN (From Scratch)
Evaluates the trained from-scratch 3D ViT-FPN model under physical 3D Rician scanner noise ($\sigma=0.08$), quadratic RF $B_1$ field bias, and missing sequence triage (T1c-only, FLAIR-only).


In [ ]:
!python scripts/evaluate_ood_3d.py \
    --model_type visreg_jepa \
    --from_scratch \
    --seed {SEED} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp


## 9. Export & Package Artifacts
Packages from-scratch 3D ViT-FPN checkpoints, training curves, and all evaluation benchmark summaries into `vit_scratch_outputs.zip`.


In [ ]:
import datetime
import time
from pathlib import Path

from brats_jepa_3d.config import IN_KAGGLE, PROJECT_ROOT
from brats_jepa_3d.utils import export_artifacts

# 1. Package trained from-scratch ViT checkpoints, metrics, and logs into a verified zip archive
base_working = Path("/kaggle/working") if IN_KAGGLE else PROJECT_ROOT
export_res = export_artifacts(
    export_name="vit_scratch_outputs",
    export_dir=base_working / "export_vit_scratch",
    zip_path=base_working / "vit_scratch_outputs.zip",
    model_prefix="scratch",
    verbose=True,
)

# 2. Session Timing Report
NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
